<a href="https://colab.research.google.com/github/BDH-teacher/RL_from_basics/blob/main/RL_from_basic_ch_9_REINFORCE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REINFORCE 구현

## 라이브러리 import 및 하이퍼 파라미터 정의

In [1]:
# 최신버전으로 수정

!pip install gym pyvirtualdisplay > /dev/null 2>&1
!pip install gymnasium[classic-control] > /dev/null 2>&1

In [2]:
import base64
import collections
import glob
import io
import random

import gymnasium as gym # 최신버전으로 수정
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

from IPython import display as ipythondisplay
from IPython.display import HTML

In [3]:
# Hyperparameters

learning_rate = 0.0002
gamma         = 0.98

## 정책 네트워크 클래스

In [4]:
class Policy(nn.Module):
    def __init__(self):
        super(Policy, self).__init__()
        self.data = []

        self.fc1 = nn.Linear(4, 128)
        self.fc2 = nn.Linear(128, 2)
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.softmax(self.fc2(x), dim=0)
        return x

    def put_data(self, item):
        self.data.append(item)

    def train_net(self):
        R = 0
        self.optimizer.zero_grad()

        for r, prob in self.data[::-1]:
            R = r + gamma * R
            loss = -R * torch.log(prob)
            loss.backward()

        self.optimizer.step()
        self.data = []

- loss = -torch.log(prob) * R이 $-G_t * \log \pi_\theta(s_t, a_t)$와 동일함
   - main 함수에서 저장한 데이터를 통해 업데이트를 진행함
   - prob 변수에는 $\pi_\theta(s_t, a_t)$값이 담겨있기 때문에 prob에 log를 취하고 그 앞에 -R을 곱하여 loss를 계산함 (R은 리턴) <br/><br/>
- 모든 데이터에 대해 loss를 계산하여 backward함수를 호출하면 그라디언트가 계산됨 <br/><br/>
- for문이 끝나고 optimizer.step이 실행되면 축적된 그라디언트를 통해 뉴럴넷의 파라미터가 업데이트 됨

## 메인 함수

In [5]:
def main():
    env = gym.make('CartPole-v1')
    pi = Policy()
    score = 0.0
    print_interval = 20

    max_score = 0.0

    for n_epi in range(5000):
        s, info = env.reset()
        done = False

        while not done: # CartPole-v1 forced to terminates at 500 step.
            prob = pi(torch.from_numpy(s).float())
            m = Categorical(prob)
            a = m.sample()

            s_prime, r, terminated, truncated, info = env.step(a.item())
            done = terminated or truncated

            # terminated(진짜 종료)만 terminal로 취급하고,
            # truncated(시간제한 등)는 부트스트랩 가능하게 done_mask=1로 두는 게 일반적임
            done_mask = 0.0 if terminated else 1.0

            pi.put_data((r,prob[a]))
            s = s_prime
            score += r

        pi.train_net()

        if n_epi % print_interval == 0 and n_epi != 0:
            print("# of episode :{}, avg score : {}".format(n_epi, score/print_interval))

            if score > max_score:
                print(f'>>>> save reinforce.pth: {score:.1f}')
                torch.save(pi.state_dict(), 'reinforce.pth')
                max_score = score

            score = 0.0

    env.close()

In [6]:
main()

# of episode :20, avg score : 21.1
>>>> save reinforce.pth: 422.0
# of episode :40, avg score : 23.35
>>>> save reinforce.pth: 467.0
# of episode :60, avg score : 22.5
# of episode :80, avg score : 28.95
>>>> save reinforce.pth: 579.0
# of episode :100, avg score : 25.85
# of episode :120, avg score : 24.5
# of episode :140, avg score : 28.8
# of episode :160, avg score : 38.15
>>>> save reinforce.pth: 763.0
# of episode :180, avg score : 25.4
# of episode :200, avg score : 28.25
# of episode :220, avg score : 43.65
>>>> save reinforce.pth: 873.0
# of episode :240, avg score : 40.65
# of episode :260, avg score : 29.5
# of episode :280, avg score : 39.05
# of episode :300, avg score : 42.7
# of episode :320, avg score : 40.35
# of episode :340, avg score : 49.25
>>>> save reinforce.pth: 985.0
# of episode :360, avg score : 45.2
# of episode :380, avg score : 37.6
# of episode :400, avg score : 39.9
# of episode :420, avg score : 43.45
# of episode :440, avg score : 41.6
# of episode :4

- 정책 네트워크 pi가 계산한 각 액션별 확률에 m.sample 함수를 이용해 하나의 액션을 샘플링 하는 방식으로 액션을 선택함
   - 샘플링된 액션을 실제로 실행하면 환경에서는 상태 전이가 일어나고, 다음 상태와 보상, 에피소드가 끝난는지 여부 등을 관측함 <br/><br/>
- RINFORCE 알고리즘에서는 $\pi_{\theta}(s_t,a_t)$와 $G_t$만 있으면 loss를 계산할 수 있기 때문에 확률값 prob[a]와 보상 r을 데이터에 저장함
   - 이후 에피소드가 끝나면 pi.train_net()을 통해 한 에피소드 동안 모은 데이터를 이용해 실제 업데이트가 이루어짐

## 실행

In [7]:
#import shutil
#import os

env = gym.make('CartPole-v1', render_mode='rgb_array')
pi = Policy()
pi.load_state_dict(torch.load('reinforce.pth'))

# Remove existing video directory to ensure a fresh recording
#if os.path.exists('./video_reinforce'):
#    shutil.rmtree('./video_reinforce')

# Recreate the environment wrapper for recording. Set episode_trigger to record every episode.
env = gym.wrappers.RecordVideo(env, './video_reinforce')

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


In [8]:
s, info = env.reset()
done = False

while not done:
    prob = pi(torch.from_numpy(s).float())
    m = Categorical(prob)
    action = m.sample()
    s_prime, r, terminated, truncated, info = env.step(action.item())
    done = terminated or truncated
    s = s_prime
    print(action.item(), r)

env.close()

1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
0 1.0
1 1.0
1 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
0 1.0
0 1.0
1 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
0 1.0
1 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.

In [9]:
# play recorded video
def show_video():
    mp4list = glob.glob('video_reinforce/*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        ipythondisplay.display(HTML(data='''
            <video alt="test" autoplay loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
            </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

In [10]:
show_video()